# Day 2 — Retrieval Optimization
### AI Clinical Decision Support Lite Hackathon · Plan B

**Prepared by the Day 2 Notebook Council** (see `notebooks/COUNCIL.md` for reviewer credits)

Day 1 built an index that returns *something*. Today is about proving it returns the
*right* thing — with real, measured numbers instead of a single example query.

**By the end of this notebook you will be able to:**
1. Explain how `top_k` trades off precision against coverage
2. Run a controlled experiment comparing chunk-size configurations
3. Build a small test set and compute Retrieval Precision@k by hand
4. Read your own results and decide what to change before Day 3

> This notebook rebuilds the Day 1 index at the top so it's self-contained — you can run
> it independently without re-running `Day1_Document_Ingestion.ipynb` first.


## 0. Setup — Rebuild the Day 1 Index


In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))

import config
from ingest import load_pdfs, chunk_documents, build_index
from query import load_index, retrieve

pages = load_pdfs(config.DATA_DIR)
chunks = chunk_documents(pages)
vectordb = build_index(chunks)
print(f"\nIndex ready: {len(chunks)} chunks from {len(pages)} pages.")



Index ready: 134 chunks from 74 pages.


## 1. What `top_k` Actually Controls

`top_k` is the number of chunks retrieval hands to the generation step. It is not a minor
setting — it's your first real trade-off of the day:

| `k` | Effect | Risk |
|---|---|---|
| Too low (1–2) | Very focused | Misses relevant evidence sitting in another section |
| Balanced (3–5) | Good coverage, manageable context | Usually the right starting point |
| Too high (10+) | Broad coverage | Dilutes context, invites irrelevant or contradictory chunks |

Let's see this directly: run the same question at three different `k` values and compare.


In [2]:
question = "What is the target blood pressure for a patient with cardiovascular disease?"

for k in [1, 3, 8]:
    results = retrieve(vectordb, question, k=k)
    print(f"--- k={k} ---")
    for doc, score in results:
        print(f"  score={score:.3f}  page {doc.metadata.get('page_number')}: "
              f"{doc.page_content[:70].strip()}...")
    print()


--- k=1 ---
  score=0.789  page 28: GUIDELINE FOR THE PHARMACOLOGICAL TREATMENT OF HYPERTENSION IN ADULTS...

--- k=3 ---
  score=0.789  page 28: GUIDELINE FOR THE PHARMACOLOGICAL TREATMENT OF HYPERTENSION IN ADULTS...
  score=0.770  page 9: WHO recommends a target systolic blood pressure treatment goal of <130...
  score=0.740  page 11: WHO recommends a target systolic blood pressure treatment goal of <130...

--- k=8 ---
  score=0.789  page 28: GUIDELINE FOR THE PHARMACOLOGICAL TREATMENT OF HYPERTENSION IN ADULTS...
  score=0.770  page 9: WHO recommends a target systolic blood pressure treatment goal of <130...
  score=0.740  page 11: WHO recommends a target systolic blood pressure treatment goal of <130...
  score=0.695  page 10: no data regarding target BP or the best antihypertensive agent to trea...
  score=0.690  page 13: 1 Introduction
More people die each year from cardiovascular diseases...
  score=0.686  page 2: Executive summary
More people die each year from cardiovascular

### Checkpoint 1

Look at the `k=8` output. Are all 8 results still genuinely about target blood pressure, or
do the later ones start drifting into unrelated sections of the guideline? This drift —
not an error, just noise — is exactly why `top_k` needs to be tuned deliberately rather
than set high "to be safe."


## 2. Ablation Experiment — Chunk Size & Overlap

A proper experiment needs a fixed method: same source, same queries, only the chunking
configuration changes. We'll rebuild the index three times with different `chunk_size` /
`chunk_overlap` pairs and compare retrieval on a fixed query set.


In [3]:
import importlib
from langchain_text_splitters import RecursiveCharacterTextSplitter
from ingest import get_embedding_function
from langchain_chroma import Chroma

test_queries = [
    "What blood pressure threshold should trigger starting medication?",
    "What are the three recommended first-line drug classes?",
    "Can nurses or pharmacists prescribe antihypertensive treatment?",
]

configurations = [
    {"name": "Small (200/0)",    "chunk_size": 200,  "chunk_overlap": 0},
    {"name": "Balanced (400/50)", "chunk_size": 400,  "chunk_overlap": 50},
    {"name": "Large (600/100)",  "chunk_size": 600,  "chunk_overlap": 100},
]

embed_fn = get_embedding_function()
experiment_results = []

for cfg in configurations:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=cfg["chunk_size"] * 4,
        chunk_overlap=cfg["chunk_overlap"] * 4,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    test_chunks = splitter.split_documents(pages)
    test_db = Chroma.from_documents(
        documents=test_chunks, embedding=embed_fn,
        collection_name=f"experiment_{cfg['chunk_size']}",
    )

    avg_score = 0
    for q in test_queries:
        results = test_db.similarity_search_with_relevance_scores(q, k=3)
        avg_score += sum(s for _, s in results) / len(results)
    avg_score /= len(test_queries)

    experiment_results.append({"config": cfg["name"], "n_chunks": len(test_chunks), "avg_top3_score": avg_score})
    print(f"{cfg['name']:<20} chunks={len(test_chunks):>4}   avg top-3 relevance={avg_score:.3f}")


Small (200/0)        chunks= 282   avg top-3 relevance=0.699
Balanced (400/50)    chunks= 163   avg top-3 relevance=0.680
Large (600/100)      chunks= 119   avg top-3 relevance=0.662


### Checkpoint 2

This is a real experiment, not a demonstration — the numbers above come from your actual
index, actual embedding model, and actual test queries. Before moving on:

- Which configuration scored highest on average?
- Did the configuration with the *most* chunks also score the *best*? (It often doesn't —
  more chunks means more noise to filter through, not automatically better retrieval.)

Record which configuration you're keeping in `config.py` and why — you'll want that
justification ready when a judge asks about it on Day 5.


## 3. Build Your Test Set

A single example query proves nothing. `eval/Day2_Evaluation_Test_Set.csv` — already in
your starter kit — has 8 real questions with verified expected sources. Let's load it and
use it to compute a real metric.


In [4]:
import csv

test_set = []
with open("./eval/Test_Set.csv", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        test_set.append(row)

print(f"Loaded {len(test_set)} test questions.\n")
for row in test_set[:3]:
    print("Q:", row["Question"])
    print("   Expected:", row["Expected Source (Document / Section / Page)"])
    print()


Loaded 8 test questions.

Q: What is the target blood pressure for a patient with cardiovascular disease?
   Expected: WHO Hypertension Guideline 2021 / Section 3.6 / Page 28

Q: What blood pressure threshold should trigger starting medication?
   Expected: WHO Hypertension Guideline 2021 / Section 3.1 / Page 19

Q: What are the three recommended first-line drug classes?
   Expected: WHO Hypertension Guideline 2021 / Section 3.4 / Page 24



## 4. Compute Retrieval Precision@k

$$\text{Precision@k} = \frac{\text{relevant chunks in top-k}}{k}$$

For each question, we check whether the retrieved chunks' page numbers match the expected
page from the test set. This is a simplified, page-level version of Precision@k — good
enough to get a real, defensible number today.


In [5]:
def page_matches_expected(retrieved_page, expected_text):
    """Very simple check: does the expected 'Page N' appear, and does our retrieved
    page number match it? Returns None for the deliberate out-of-scope question."""
    import re
    m = re.search(r"Page (\d+)", expected_text)
    if not m:
        return None  # out-of-scope control question — handled separately
    expected_page = int(m.group(1))
    return retrieved_page == expected_page


k = 3
precisions = []
print(f"{'Question':<55} {'P@' + str(k):<8} Notes")
print("-" * 90)

for row in test_set:
    expected = row["Expected Source (Document / Section / Page)"]
    if "Not covered" in expected:
        print(f"{row['Question'][:53]:<55} {'N/A':<8} out-of-scope control question")
        continue

    results = retrieve(vectordb, row["Question"], k=k)
    hits = sum(
        1 for doc, _ in results
        if page_matches_expected(doc.metadata.get("page_number"), expected)
    )
    precision = hits / k
    precisions.append(precision)
    print(f"{row['Question'][:53]:<55} {precision:<8.2f}")

avg_precision = sum(precisions) / len(precisions)
print("-" * 90)
print(f"Average Precision@{k} across {len(precisions)} scored questions: {avg_precision:.2f}")


Question                                                P@3      Notes
------------------------------------------------------------------------------------------
What is the target blood pressure for a patient with    0.33    
What blood pressure threshold should trigger starting   0.33    
What are the three recommended first-line drug classe   0.00    
Can nurses or pharmacists prescribe antihypertensive    0.33    
What laboratory tests are recommended before starting   0.00    
When should cardiovascular disease risk assessment be   0.00    
How frequently should blood pressure be reassessed af   0.67    
What is the recommended dietary salt intake for healt   N/A      out-of-scope control question
------------------------------------------------------------------------------------------
Average Precision@3 across 7 scored questions: 0.24


## 5. Embedding Model Comparison

Everything above used a single embedding model — whatever `config.EMBEDDING_MODEL`
is set to. But the embedding model is itself a retrieval-quality lever, exactly like
`chunk_size` was in Section 2. This section builds the index **twice**, once per model,
with the winning chunking configuration from Section 2 held fixed, and scores both on
the same Precision@k benchmark from Section 4. Same method as the chunking ablation:
one variable changes, everything else stays fixed.

We compare:

| Model | Dim | Notes |
|---|---|---|
| `BAAI/bge-small-en-v1.5` | 384 | Current `config.py` default — fast, low memory |
| `BAAI/bge-base-en-v1.5` | 768 | Larger sister model — typically higher retrieval quality, slower to embed, ~3x the index size |

Both are supported directly by `FastEmbedEmbeddings` (via `ingest.get_embedding_function`),
so no new dependency is needed — just a different `model_name` string.


In [6]:
from ingest import get_embedding_function, chunk_documents

# Use the winning chunk_size / chunk_overlap you identified in Section 2.
# These currently mirror config.py's defaults — update if your ablation picked differently.
BEST_CHUNK_SIZE = config.CHUNK_SIZE
BEST_CHUNK_OVERLAP = config.CHUNK_OVERLAP

embedding_models = [
    "BAAI/bge-small-en-v1.5",   # current default
    "BAAI/bge-base-en-v1.5",    # comparison candidate
]

fixed_chunks = chunk_documents(pages, chunk_size=BEST_CHUNK_SIZE, chunk_overlap=BEST_CHUNK_OVERLAP)
print(f"Using {len(fixed_chunks)} chunks (size={BEST_CHUNK_SIZE}, overlap={BEST_CHUNK_OVERLAP}) for both models.\n")

model_indexes = {}
for model_name in embedding_models:
    print(f"Building index with {model_name} ...")
    embed_fn = get_embedding_function(model_name)
    safe_name = model_name.replace('/', '_').replace('.', '_')
    model_db = Chroma.from_documents(
        documents=fixed_chunks,
        embedding=embed_fn,
        collection_name=f"embed_compare_{safe_name}",
    )
    model_indexes[model_name] = model_db
    print(f"  done — {len(fixed_chunks)} chunks embedded.\n")

Using 134 chunks (size=512, overlap=64) for both models.

Building index with BAAI/bge-small-en-v1.5 ...
  done — 134 chunks embedded.

Building index with BAAI/bge-base-en-v1.5 ...
  done — 134 chunks embedded.



In [7]:
# Reuse the exact same page_matches_expected() + test_set from Section 4 so the
# two models are scored on an identical benchmark, identical k, identical scoring logic.

k = 3
model_precisions = {}

for model_name, model_db in model_indexes.items():
    precisions = []
    for row in test_set:
        expected = row["Expected Source (Document / Section / Page)"]
        if "Not covered" in expected:
            continue

        results = model_db.similarity_search_with_relevance_scores(row["Question"], k=k)
        hits = sum(
            1 for doc, _ in results
            if page_matches_expected(doc.metadata.get("page_number"), expected)
        )
        precisions.append(hits / k)

    model_precisions[model_name] = sum(precisions) / len(precisions)

print(f"{'Model':<28} {'Avg Precision@' + str(k):<15}")
print("-" * 45)
for model_name, avg_p in model_precisions.items():
    print(f"{model_name:<28} {avg_p:<15.2f}")

best_model = max(model_precisions, key=model_precisions.get)
print(f"\nBest performing model on this benchmark: {best_model} (Precision@{k} = {model_precisions[best_model]:.2f})")


Model                        Avg Precision@3
---------------------------------------------
BAAI/bge-small-en-v1.5       0.24           
BAAI/bge-base-en-v1.5        0.33           

Best performing model on this benchmark: BAAI/bge-base-en-v1.5 (Precision@3 = 0.33)


### Checkpoint 4

- Did the larger model (`bge-base-en-v1.5`) actually score higher, or did it just cost
  more compute for the same precision? Bigger embedding models aren't automatically better
  retrievers on a small, domain-specific benchmark like this one.
- If the two scores are close, that's a real result too — it means `config.py` can keep
  the smaller/faster model without giving up measurable retrieval quality.
- Update `config.py`'s `EMBEDDING_MODEL` only if the winning model's precision gain is
  large enough to justify the extra index size and embedding latency — record that
  trade-off explicitly, the same way you recorded the chunking decision in Checkpoint 2.


### Checkpoint 3 — Day 2 Self-Check

- [ ] You ran the same query at 3 different `k` values and can explain the trade-off out loud
- [ ] You ran a real ablation experiment across 3 chunking configurations and picked one
- [ ] You have an actual Precision@k number — not a guess — for your current index
- [ ] `config.py` reflects the configuration you're keeping, and you know why
- [ ] You compared at least 2 embedding models on the same Precision@k benchmark and can justify which one you kept

For extra rigor, open `templates/Day2_Retrieval_Scorecard_Template.xlsx` and log these
same numbers there — it has live formulas so your team average recalculates automatically
as more teammates fill it in.

## What's Next

Day 3's notebook assumes retrieval is now trustworthy — it moves on to constraining the
model so tightly that every generated answer can only say what these retrieved chunks
actually support.
